# A closer look at Code as Policies with CaP-X

<p align="center">
  <img src="images/cap_thumb.png" width="900">
</p>

[CaP-X](https://github.com/capgym/cap-x) (*Code-as-Policies eXtended*) turns a natural-language manipulation task into an executable Python policy. A locally served model writes a short program that composes grounded perception and robot-control primitives; CaP-X runs it on a Franka Panda in Robosuite/MuJoCo and scores the resulting episode.

## Goals

* Compare CaP-X's generated program with the step-by-step tool loop used by RAI
* Follow a task from words through perception and grasp planning to joint motion and reward
* Read the system prompt, the primitive API, the generated code, and the live `goto_pose` source
* Generate a policy, run it with video, then run three fresh rollouts on a harder task
* Tell sequencing and geometry errors apart from failures in the grounded services

## RAI vs CaP-X

Both approaches map language to robot motion, but the model has a different job.

| | RAI | CaP-X |
| --- | --- | --- |
| Model output | one tool call at a time | one Python program up front |
| Model position | inside a step-by-step control loop | upstream of program execution |
| Runtime driver | agent graph | `env.step(program)` and CPython |
| Grounding | services exposed as tools | services exposed as Python primitives |
| Evaluation | inspect the interaction | reward and success over seeded rollouts |
| Typical failure | a bad next tool call | syntax/runtime error or incorrect geometry |

Neither system asks the language model to implement perception, grasp planning, IK, or physics. In CaP-X it writes the sequencing and geometry around an API that calls those services.

## Pipeline overview

1. A **task prompt** names the goal and exposes the available primitive signatures and docstrings.
2. The local **Lemonade LLM** generates executable Python that calls those primitives.
3. `env.step(program)` executes the policy with bound `FrankaControlApi` functions.
4. A perception call uses **OWLv2** to ground an object phrase into boxes, then **SAM2** to turn a selected box into a mask.
5. Masked depth supports a 3D point cloud and **oriented bounding box (OBB)** pose; for grasping, **Contact-GraspNet** proposes ranked 6-DoF grasp poses.
6. `goto_pose` sends the requested pose to **PyRoKi**, which solves inverse kinematics for robot joints.
7. **Robosuite/MuJoCo** executes the joint and gripper commands, after which CaP-X reports reward, completion, errors, and video.

## Serve Gemma locally

CaP-X only requires an OpenAI-compatible chat-completions endpoint. `ensure_lemonade` starts the local Lemonade daemon if needed and loads the image-cached Gemma E4B model. Setup time is recorded separately from generation and robot rollout time.

The same model settings are used for the walkthrough and benchmark below: `Gemma-4-E4B-it-GGUF`, temperature `1.0`, and at most `4096` generated tokens.

In [ ]:
import os
import sys
from time import perf_counter

sys.path.insert(0, "/ryzers/notebooks/scripts")

# CaP-X writes generated policies and rollout videos under CAPX_WORK. This directory
# is a copy of projects/LocalInference, so whatever is committed there ships in the
# image beside these notebooks.
os.environ.setdefault("CAPX_WORK", "/ryzers/notebooks/outputs")

from capx_demo import (
    PRIMITIVES,
    WORK,
    analyze_program,
    benchmark_scenarios,
    clean_program,
    ensure_lemonade,
    llama_metrics,
    metric_delta,
    quiet_output,
    show_trial_grid,
    show_video,
    trial_introspection,
)

MODEL = "Gemma-4-E4B-it-GGUF"
SERVER_URL = "http://localhost:13305/api/v1/chat/completions"
TEMPERATURE = 1.0
MAX_TOKENS = 4096

SCENARIOS = {
    "cube lift": "env_configs/cube_lifting/franka_robosuite_cube_lifting.yaml",
    "cube stack": "env_configs/cube_stack/franka_robosuite_cube_stack.yaml",
    "spill wipe": "env_configs/spill_wipe/franka_robosuite_spill_wipe.yaml",
}
CONFIG_PATH = SCENARIOS["cube lift"]
LEMONADE_SECONDS = ensure_lemonade(MODEL)

## System prompt and primitive API

CaP-X sends a short system instruction plus a task-specific user message. The user message contains an `APIs:` section rendered from the live primitive signatures and docstrings.

The five main primitives are:

* `get_object_pose(description, return_bbox_extent=...)`: OWLv2 box grounding, SAM2 masking, and depth/OBB reconstruction return a 3D position, orientation, and optional full extents.
* `sample_grasp_pose(description)`: the same grounded perception path supplies Contact-GraspNet with mask and depth so it can return a candidate 6-DoF grasp.
* `goto_pose(position, quaternion_wxyz, z_approach=...)`: PyRoKi solves IK and the environment executes the resulting arm motion.
* `open_gripper()`: commands the gripper open.
* `close_gripper()`: commands the gripper closed.

`home_pose()` is also available as a safe-motion helper.

The next cell starts those services and constructs the same environment that the CaP-X launcher would create. Both come from the task config at `/ryzers/cap-x/env_configs/cube_lifting/franka_robosuite_cube_lifting.yaml`, which names the API to bind and the services to launch.

In [ ]:
service_started = perf_counter()
with quiet_output() as service_log:
    from capx.envs.configs.instantiate import instantiate
    from capx.envs.launch import LaunchArgs
    from capx.envs.runner import _start_api_servers
    from capx.utils.launch_utils import _load_config

    args = LaunchArgs(
        config_path=CONFIG_PATH,
        model=MODEL,
        server_url=SERVER_URL,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    env_factory, config, api_servers = _load_config(args)
    servers = _start_api_servers(api_servers, 900.0)
    env = instantiate(env_factory)
    api = next(iter(env._apis.values()))
    obs, _ = env.reset(options={"trial": 0}, seed=0)

SERVICE_SECONDS = perf_counter() - service_started
SERVICES = [
    server["_target_"].split(".")[-2].removeprefix("launch_").removesuffix("_server")
    for server in api_servers
]
print(
    f"Ready in {SERVICE_SECONDS:.1f}s: {type(env).__name__} + "
    f"{type(api).__name__} (details: {service_log})"
)
print(f"Services: {', '.join(SERVICES)}")

## The prompt before generation

`obs["full_prompt"]` holds the exact two-message conversation that goes to Lemonade. The system message asks for directly executable Python. The user message supplies the task, its constraints, and the API documentation generated from the configured environment.

The prompt contains no camera image, object coordinate, or joint state. The model writes a scene-independent program; object facts enter later when that program calls grounded primitives at runtime.

In [ ]:
system_message, user_message = obs["full_prompt"]

print(system_message["content"])
print(user_message["content"][0]["text"])

## Generate the Python policy

`ModelQueryArgs` mirrors the model fields in `LaunchArgs`, and `query_model` makes one chat-completions request to the local server. CaP-X extracts the first Python block as `program`.

In [ ]:
from capx.llm.client import ModelQueryArgs, query_model
from capx.utils.launch_utils import _extract_code

query_args = ModelQueryArgs(
    model=MODEL,
    server_url=SERVER_URL,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)

metrics_before = llama_metrics()
query_started = perf_counter()
with quiet_output():
    response = query_model(query_args, obs["full_prompt"])
LLM_WALL_SECONDS = perf_counter() - query_started
LLM_METRICS = metric_delta(metrics_before, llama_metrics())

blocks = _extract_code(response["content"])
assert blocks, f"no code in the reply - raise MAX_TOKENS?\n{response['content'][-500:]}"
program, stray_tokens = clean_program(blocks[0])
if stray_tokens:
    print(f"Stripped stray decoding tokens: {sorted(set(stray_tokens))}\n")

print(
    f"LLM: {LLM_WALL_SECONDS:.1f}s wall · "
    f"{LLM_METRICS.get('prompt_tokens_total', 0):.0f} prompt tokens · "
    f"{LLM_METRICS.get('tokens_predicted_total', 0):.0f} generated tokens"
)
print(program)

## Execute the generated policy

A traceback means a Python or execution failure, a grounding message points at perception or the chosen noun phrase, and a clean low-reward run usually means bad sequencing or geometry.

Two generation artifacts show up here, and neither is a reasoning failure.

Gemma occasionally appends a stray control token such as `<unused56>` to the end of a code block, which reaches the sandbox as a `SyntaxError` before the robot moves at all. That is a decoding artifact rather than something the model wrote, so the cell above strips it and reports it when it happens.

The second one is left alone. Gemma sometimes calls `numpy.array` without emitting `import numpy`, raising a `NameError` partway through an otherwise sound policy. That is the model forgetting a line, so it stands as written. Regenerate and rerun.

In [ ]:
program_analysis = analyze_program(program)

print("Primitive calls")
for primitive in PRIMITIVES:
    print(f"  {primitive:<20} {program_analysis['primitive_calls'][primitive]}")

print("\nProgram flags")
for key in (
    "syntax_error",
    "perception_calls",
    "planner_calls",
    "uses_bbox_extent",
    "uses_approach_offset",
    "nested_pose_indexing",
    "line_count",
):
    print(f"  {key:<22} {program_analysis[key]}")

In [ ]:
env.enable_video_capture(True, clear=True)

rollout_started = perf_counter()
with quiet_output():
    _, reward, terminated, _, info = env.step(program)
ROLLOUT_SECONDS = perf_counter() - rollout_started

print(
    f"Rollout: {ROLLOUT_SECONDS:.1f}s · reward {reward:.3f} · "
    f"solved {info['task_completed']}"
)
if info["sandbox_rc"]:
    print("\nTraceback (tail):\n" + info["stderr"][-1200:])

(WORK / "cube_lift_policy.py").write_text(program)
show_video(env)

## Where the motion happens

The rollout crosses the policy/service boundary at `goto_pose`. The generated program supplies a Cartesian gripper-tip pose and optional approach distance; the primitive converts that request into a robot configuration and blocks until Robosuite/MuJoCo has executed the motion.

In the source below, `goto_pose` applies the tool-center-point offset in the end-effector frame, optionally creates a standoff waypoint for `z_approach`, and calls `ik_solve_fn` for that waypoint and the final pose. The configured solver is the PyRoKi service. When available, the previous configuration is supplied to keep successive solutions connected, and the resulting seven arm joints go to `move_to_joints_blocking` for execution.

A syntactically valid program can still request an awkward or unreachable Cartesian target. The static program flags show what the LLM asked for, and the rollout shows whether the grounded pose and IK result produced useful motion.

In [ ]:
import inspect

goto_pose_source = inspect.getsource(type(api).goto_pose)
print(goto_pose_source)

## Three fresh rollouts: cube stack

One successful video does not measure reliability, and cube lift is the easy case. Cube stack raises the difficulty: ground two objects, grasp one, derive a placement height from both bounding-box extents, and release accurately on top of the other.

`benchmark_scenarios` generates a fresh policy for each of three seeded trials and runs it, retaining the prompt, response, extracted program, diagnostics, reward, and video for every attempt. Expect most trials to fail.

The programs Gemma writes are structurally alike from trial to trial: ground both cubes, sample one grasp, three moves, close and open. The rewards still range from near zero to a clean solve. Two things vary alongside the code. Each trial reseeds the scene, so the cube layout differs, and the grasp planner downsamples its input point cloud stochastically, so one scene need not produce one grasp. Notebook 4 returns to this task and evolves a policy against it.

In [ ]:
BENCHMARK_SCENARIOS = {"cube stack": SCENARIOS["cube stack"]}
BENCHMARK_TRIALS = 3

benchmark_started = perf_counter()
benchmark_results = benchmark_scenarios(
    model=MODEL,
    server_url=SERVER_URL,
    scenarios=BENCHMARK_SCENARIOS,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    trials=BENCHMARK_TRIALS,
)
BENCHMARK_SECONDS = perf_counter() - benchmark_started
print(f"\n{BENCHMARK_TRIALS} complete trials: {BENCHMARK_SECONDS:.1f}s wall")

introspection_rows = []
for scenario in BENCHMARK_SCENARIOS:
    scenario_trials = [
        trial for trial in benchmark_results if trial["label"] == scenario
    ]
    print(f"\n{scenario}: {len(scenario_trials)} rollout videos")
    show_trial_grid(scenario_trials)
    for row in trial_introspection(scenario_trials):
        introspection_rows.append({"scenario": scenario, **row})

print("\nGenerated-policy evidence")
print(
    f"{'scenario':<12} {'seed':>4} {'outcome':<18} {'reward':>7} "
    f"{'calls':>5} {'per':>3} {'plan':>4} {'bbox':>5} {'approach':>8} {'nested':>6}"
)
for row in introspection_rows:
    print(
        f"{row['scenario']:<12} {row['trial']:>4} {row['outcome']:<18} "
        f"{row['reward']:>7.3f} {row['primitive_calls']:>5} "
        f"{row['perception_calls']:>3} {row['planner_calls']:>4} "
        f"{str(row['bbox_extent']):>5} {str(row['approach_offset']):>8} "
        f"{str(row['nested_pose_indexing']):>6}"
    )
    if row["syntax_error"]:
        print(f"  syntax error: {row['syntax_error']}")

## Key takeaways

* **Code is the policy.** The LLM emits one executable program whose control flow, object phrases, waypoints, offsets, and geometry determine behavior.
* **Grounded systems provide the operations.** OWLv2 finds boxes, SAM2 produces masks, depth/OBB reconstruction estimates 3D pose and size, Contact-GraspNet proposes grasps, and PyRoKi maps Cartesian requests to robot joints.
* **The LLM does sequencing and error-prone logic.** It decides which primitives to call and performs tuple unpacking, coordinate arithmetic, orientation reuse, and failure-sensitive ordering.
* **The simulator supplies the evidence.** Robosuite/MuJoCo execution turns a plausible-looking program into reward, diagnostics, and video.
* **Repeated rollouts measure the whole stack.** Structurally similar cube stack policies can earn very different rewards, since each trial reseeds the layout and the grasp planner samples its own input. A success rate scores perception, grasp planning, and physics along with the generated code.

## What to try next

* Change only the natural-language task and compare the resulting primitive calls with `analyze_program`.
* Inspect `introspection_rows` to compare two policies that received different rewards on the same scenario.
* Run `benchmark_scenarios(..., oracle=True)` as a reference when separating generated-policy errors from environment or service issues.
* Point `SERVER_URL` at another OpenAI-compatible backend and keep the rest of the CaP-X pipeline unchanged.

## References

* [CaP-X](https://github.com/capgym/cap-x)
* [OWLv2 Large](https://huggingface.co/google/owlv2-large-patch14-ensemble) · [SAM2.1 Large](https://huggingface.co/facebook/sam2.1-hiera-large) · [Contact-GraspNet](https://github.com/NVlabs/contact_graspnet) · [PyRoKi](https://github.com/chungmin99/pyroki)
* [Robosuite](https://github.com/ARISE-Initiative/robosuite) · [MuJoCo](https://mujoco.org/)
* [Lemonade](https://lemonade-server.ai/)